# 📷 Feature Extraction Kamera Proctoring — v3 FIXED

## Perbaikan dari versi sebelumnya:
| Masalah | Penyebab | Perbaikan |
|---|---|---|
| Pitch/Yaw ribuan derajat | `RQDecomp3x3` × 360 salah | Ganti dengan `arcsin/arctan2` dari rotation matrix |
| Distracted rate 99,7% | Head pose tidak valid | Setelah fix, Yaw kembali ke rentang ±90° |
| Focused hanya 0,1% | Efek dari bug di atas | Akan normal setelah fix |

## Tidak perlu GPU — CPU sudah cukup untuk dlib
**Runtime → Change runtime type → None (CPU)**

## LANGKAH 1: Mount Drive & Instalasi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive terhubung!')

In [ ]:
!pip install dlib scipy -q

import os, re, time, warnings
import pandas as pd
import numpy as np
import cv2
import dlib
from datetime import datetime
from scipy.spatial import distance as dist
from tqdm.notebook import tqdm
warnings.filterwarnings('ignore')

print(f'✅ dlib: {dlib.__version__} | OpenCV: {cv2.__version__}')

## LANGKAH 2: Konfigurasi

In [ ]:
# ══════════════════════════════════════════
# SESUAIKAN PATH INI
# ══════════════════════════════════════════
PHOTO_DIR  = '/content/drive/MyDrive/ITB/Data Disertasi/Log Data LMS/proctoring_data'
OUTPUT_CSV = '/content/drive/MyDrive/ITB/Data Disertasi/Log Data LMS/camera_features_real_v3.csv'
MAX_PER_USER = None  # None = proses semua
# ══════════════════════════════════════════

os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)

if os.path.exists(PHOTO_DIR):
    all_files = [f for f in os.listdir(PHOTO_DIR)
                 if f.lower().endswith(('.jpg','.jpeg','.png'))]
    print(f'✅ Folder: {PHOTO_DIR}')
    print(f'   Total file: {len(all_files):,}')
else:
    print(f'❌ Folder tidak ditemukan: {PHOTO_DIR}')

## LANGKAH 3: Download Model dlib

In [ ]:
import bz2, urllib.request

model_dat = '/content/shape_predictor_68_face_landmarks.dat'
model_bz2 = '/content/shape_predictor_68_face_landmarks.dat.bz2'

if not os.path.exists(model_dat):
    print('Mendownload model dlib (99MB)...')
    urllib.request.urlretrieve(
        'http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2',
        model_bz2)
    with bz2.open(model_bz2,'rb') as fi, open(model_dat,'wb') as fo:
        fo.write(fi.read())
    os.remove(model_bz2)
    print('✅ Model berhasil didownload!')
else:
    print('✅ Model sudah tersedia!')

## LANGKAH 4: Fungsi Ekstraksi — HEAD POSE FIXED

In [ ]:
# ── Inisialisasi dlib ──
detector  = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor(model_dat)

# ── Landmark indices (68 titik) ──
LEFT_EYE  = list(range(42, 48))
RIGHT_EYE = list(range(36, 42))
MOUTH     = list(range(48, 68))
HP_IDX    = [30, 8, 36, 45, 48, 54]
HP_3D     = np.array([
    [0.0,     0.0,    0.0],
    [0.0,  -330.0,  -65.0],
    [-225.0, 170.0, -135.0],
    [225.0,  170.0, -135.0],
    [-150.0,-150.0, -125.0],
    [150.0, -150.0, -125.0]
], dtype=np.float64)


def shape_to_np(shape):
    return np.array([[shape.part(i).x, shape.part(i).y]
                     for i in range(68)], dtype=np.float64)

def calc_ear(pts, idx):
    p = pts[idx]
    return (dist.euclidean(p[1],p[5]) + dist.euclidean(p[2],p[4])) / \
           (2.0 * dist.euclidean(p[0],p[3]) + 1e-6)

def calc_mar(pts, idx):
    p = pts[idx]
    return (dist.euclidean(p[2],p[10]) + dist.euclidean(p[4],p[8])) / \
           (2.0 * dist.euclidean(p[0],p[6]) + 1e-6)

def calc_head_pose(pts, w, h):
    """Head pose via PnP — FIXED: gunakan arcsin/arctan2 bukan RQDecomp3x3 × 360"""
    pts2d = pts[HP_IDX].astype(np.float64)
    cam   = np.array([[w,0,w/2],[0,w,h/2],[0,0,1]], dtype=np.float64)
    ok, rvec, _ = cv2.solvePnP(HP_3D, pts2d, cam, np.zeros((4,1)),
                                flags=cv2.SOLVEPNP_ITERATIVE)
    if not ok:
        return None, None, None
    rmat, _ = cv2.Rodrigues(rvec)

    # ✅ CARA BENAR: ekstrak Euler angles dari rotation matrix
    # Hasilnya dalam derajat, rentang wajar: pitch ±45°, yaw ±90°, roll ±30°
    pitch = np.degrees(np.arcsin(-np.clip(rmat[2][0], -1, 1)))
    yaw   = np.degrees(np.arctan2(rmat[1][0], rmat[0][0]))
    roll  = np.degrees(np.arctan2(rmat[2][1], rmat[2][2]))
    return float(pitch), float(yaw), float(roll)

def calc_gaze(pts, w):
    lc  = pts[LEFT_EYE].mean(axis=0)
    rc  = pts[RIGHT_EYE].mean(axis=0)
    fw  = abs(pts[16][0] - pts[0][0]) + 1e-6
    gx  = ((lc[0] + rc[0]) / 2 - w / 2) / fw
    gy  = float((lc[1] + rc[1]) / 2 - pts[30][1]) / (fw + 1e-6)
    return float(np.clip(gx, -1, 1)), float(np.clip(gy, -1, 1))

def parse_filename(filename):
    """Format: {seq}_user_{userid}_{timestamp10digit}.ext"""
    m = re.match(r'^(\d+)_user_(\d+)_(\d{10})\.(\w+)$', filename)
    if m:
        ts = int(m.group(3))
        try:
            dt = datetime.fromtimestamp(ts)
            if 2024 <= dt.year <= 2030:
                return int(m.group(1)), int(m.group(2)), ts, dt
        except: pass
    return None, None, None, None

def process_image(img_path):
    img = cv2.imread(img_path)
    if img is None: return None
    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = detector(gray, 0)

    if len(faces) == 0:
        return {'face_detected':0,
                'ear_left':None,'ear_right':None,'ear_avg':None,'is_drowsy':None,
                'mar':None,'is_yawning':None,
                'pitch':None,'yaw':None,'roll':None,
                'gaze_x':None,'gaze_y':None,'gaze_label':'unknown',
                'is_distracted':None,'engagement_label':'missing'}

    shape = predictor(gray, faces[0])
    pts   = shape_to_np(shape)

    el  = calc_ear(pts, LEFT_EYE)
    er  = calc_ear(pts, RIGHT_EYE)
    ea  = (el + er) / 2
    ma  = calc_mar(pts, MOUTH)
    pt, ya, ro = calc_head_pose(pts, w, h)
    gx, gy = calc_gaze(pts, w)

    # Gaze label
    if   abs(gx)<0.25 and abs(gy)<0.25: gl = 'center'
    elif gx < -0.25: gl = 'left'
    elif gx >  0.25: gl = 'right'
    elif gy < -0.25: gl = 'up'
    else:            gl = 'down'

    # Flags
    is_drow = int(ea  < 0.21)
    is_yawn = int(ma  > 0.60)
    is_dist = int((ya is not None and abs(ya) > 30) or gl in ['left','right'])

    # Label
    if is_drow or is_yawn: lbl = 'fatigued'
    elif is_dist:          lbl = 'distracted'
    else:                  lbl = 'focused'

    return {
        'face_detected': 1,
        'ear_left':  round(el, 4), 'ear_right': round(er, 4), 'ear_avg': round(ea, 4),
        'is_drowsy': is_drow,
        'mar':       round(ma, 4), 'is_yawning': is_yawn,
        'pitch': round(pt, 4) if pt is not None else None,
        'yaw':   round(ya, 4) if ya is not None else None,
        'roll':  round(ro, 4) if ro is not None else None,
        'gaze_x': round(gx, 4), 'gaze_y': round(gy, 4),
        'gaze_label': gl, 'is_distracted': is_dist,
        'engagement_label': lbl,
    }

# ── Verifikasi head pose fix ──
print('Verifikasi head pose fix:')
import cv2 as _cv2, numpy as _np
_rvec = _np.array([[0.1],[-0.05],[0.02]])
_rmat,_ = _cv2.Rodrigues(_rvec)
_p = _np.degrees(_np.arcsin(-_rmat[2][0]))
_y = _np.degrees(_np.arctan2(_rmat[1][0],_rmat[0][0]))
print(f'  Test pitch={_p:.1f}° yaw={_y:.1f}° → harus dalam ±90°')
assert abs(_p) < 90 and abs(_y) < 90, 'HEAD POSE MASIH SALAH!'
print('✅ Head pose fix terverifikasi — rentang normal!')
print('✅ Semua fungsi siap digunakan')

## LANGKAH 5: Scan & Parse File

In [ ]:
print('Scanning file...')
records_scan = []
n_failed = 0

for fname in all_files:
    seq, uid, ts, dt = parse_filename(fname)
    if uid is not None:
        records_scan.append({
            'filename': fname, 'seq': seq, 'user_id': uid,
            'timestamp': ts, 'datetime': dt,
            'date': dt.date(), 'hour': dt.hour,
            'day_of_week': dt.strftime('%A'),
        })
    else:
        n_failed += 1

df_scan = pd.DataFrame(records_scan)
print(f'✅ Scan selesai:')
print(f'   Berhasil parse : {len(df_scan):,} file')
print(f'   Gagal parse    : {n_failed}')
print(f'   Unique user_id : {df_scan["user_id"].nunique()}')
print(f'   Rentang waktu  : {df_scan["datetime"].min()} s/d {df_scan["datetime"].max()}')

## LANGKAH 6: Mapping user_id → Nama

In [ ]:
# Load dari file camera_features_real.csv yang sudah punya mapping
# (atau isi manual di bawah)

# Opsi A: Load mapping dari file lama
OLD_CSV = '/content/drive/MyDrive/ITB/Data Disertasi/Log Data LMS/camera_features_real.csv'
uid_to_name = {}

if os.path.exists(OLD_CSV):
    df_old = pd.read_csv(OLD_CSV)
    mapping = df_old[['user_id','nama']].drop_duplicates()
    mapping = mapping[mapping['nama'] != '-']
    uid_to_name = dict(zip(mapping['user_id'].astype(int), mapping['nama']))
    print(f'✅ Mapping dari file lama: {len(uid_to_name)} user_id → nama')
else:
    print('⚠️  File lama tidak ditemukan — mapping manual diperlukan')

# Tambahkan nama ke df_scan
df_scan['nama'] = df_scan['user_id'].map(uid_to_name).fillna(
    df_scan['user_id'].astype(str).apply(lambda x: f'User_{x}'))

print(f'\nSample mapping:')
sample = df_scan[['user_id','nama']].drop_duplicates().head(10)
print(sample.to_string(index=False))

## LANGKAH 7: Proses Batch — Ekstraksi Ulang

In [ ]:
all_records = []
stats = {'ok':0, 'detected':0, 'failed':0, 'errors':[]}
user_groups = df_scan.groupby('user_id')

print(f'Memproses {df_scan["user_id"].nunique()} mahasiswa, {len(df_scan):,} gambar...')
print(f'(Head pose fix aktif — rentang Yaw akan normal ±90°)')
print('='*55)

t_start = time.time()
uid_list = list(user_groups.groups.keys())

for uid_idx, (uid, grp) in enumerate(tqdm(user_groups, desc='Mahasiswa')):
    nama = uid_to_name.get(uid, f'User_{uid}')
    grp  = grp.sort_values('timestamp')
    if MAX_PER_USER:
        grp = grp.head(MAX_PER_USER)

    user_records = []
    try:
        for _, row in grp.iterrows():
            fpath = os.path.join(PHOTO_DIR, row['filename'])
            feat  = process_image(fpath)
            if feat is None:
                stats['failed'] += 1
                continue
            record = {
                'user_id':     uid,
                'nama':        nama,
                'filename':    row['filename'],
                'seq':         row['seq'],
                'timestamp':   row['timestamp'],
                'datetime':    row['datetime'].strftime('%Y-%m-%d %H:%M:%S'),
                'date':        row['datetime'].strftime('%Y-%m-%d'),
                'hour':        row['datetime'].hour,
                'minute':      row['datetime'].minute,
                'day_of_week': row['datetime'].strftime('%A'),
            }
            record.update(feat)
            user_records.append(record)
            stats['ok'] += 1
            if feat['face_detected']: stats['detected'] += 1

        all_records.extend(user_records)
        n = len(user_records)
        nd = sum(r['face_detected'] for r in user_records)
        print(f'  uid={uid:5d} | {nama[:30]:30s} | {n:4d} imgs | det={nd/max(n,1)*100:.0f}%')

    except Exception as e:
        stats['errors'].append({'uid':uid,'nama':nama,'err':str(e)})
        print(f'  ❌ Error uid={uid} ({nama}): {e}')

    # Auto-save setiap 10 mahasiswa
    if (uid_idx + 1) % 10 == 0 and all_records:
        pd.DataFrame(all_records).to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
        print(f'  💾 Auto-saved: {len(all_records):,} records')

# Final save
df_cam = pd.DataFrame(all_records)
df_cam.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

total_min = (time.time()-t_start)/60
print('\n' + '='*55)
print('✅ SELESAI!')
print(f'   Total diproses   : {stats["ok"]:,} gambar')
print(f'   Wajah terdeteksi : {stats["detected"]:,} ({stats["detected"]/max(stats["ok"],1)*100:.1f}%)')
print(f'   Gagal load       : {stats["failed"]}')
print(f'   Waktu total      : {total_min:.1f} menit')
print(f'   Output           : {OUTPUT_CSV}')

## LANGKAH 8: Verifikasi Hasil — Head Pose Harus Normal

In [ ]:
import matplotlib.pyplot as plt

df = pd.read_csv(OUTPUT_CSV)
df_d = df[df['face_detected']==1]

print('='*55)
print('VERIFIKASI HASIL — HEAD POSE FIXED')
print('='*55)
print(f'  Total frame       : {len(df):,}')
print(f'  Mahasiswa         : {df["user_id"].nunique()}')
print(f'  Face detected     : {df["face_detected"].mean()*100:.1f}%')

print(f'\n  Label engagement (SEHARUSNYA LEBIH SEIMBANG):')
for lbl, cnt in df['engagement_label'].value_counts().items():
    print(f'    {lbl:12s}: {cnt:6,} ({cnt/len(df)*100:.1f}%)')

print(f'\n  Head Pose (HARUS dalam rentang wajar):')
print(f'    |Yaw| mean  = {df_d["yaw"].abs().mean():.1f}° (wajar: < 30°)')
print(f'    |Pitch| mean= {df_d["pitch"].abs().mean():.1f}° (wajar: < 20°)')
print(f'    Yaw outlier (>90°): {(df_d["yaw"].abs()>90).sum()}')

print(f'\n  EAR stats:')
print(f'    Mean={df_d["ear_avg"].mean():.4f} | Drowsy={df_d["is_drowsy"].mean()*100:.1f}%')
print(f'    Distracted={df_d["is_distracted"].mean()*100:.1f}% (sebelumnya 99.7% — harus turun drastis)')

# Visualisasi distribusi Yaw
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(df_d['yaw'].dropna().clip(-90,90), bins=50, color='#028090', alpha=0.8)
axes[0].axvline(30,  color='red', ls='--', lw=2, label='Threshold ±30°')
axes[0].axvline(-30, color='red', ls='--', lw=2)
axes[0].set_title('Distribusi Yaw (FIXED)\nHarus dalam ±90°', fontweight='bold')
axes[0].set_xlabel('Yaw (derajat)'); axes[0].legend()

axes[1].hist(df_d['ear_avg'].dropna(), bins=50, color='#00A896', alpha=0.8)
axes[1].axvline(0.21, color='red', ls='--', lw=2, label='Drowsy (0.21)')
axes[1].set_title('Distribusi EAR', fontweight='bold')
axes[1].set_xlabel('EAR'); axes[1].legend()

lbl_cnt = df['engagement_label'].value_counts()
clr = {'focused':'#00A896','distracted':'#F59E0B','fatigued':'#E05A4E','missing':'#94A3B8'}
axes[2].bar(lbl_cnt.index, lbl_cnt.values,
            color=[clr.get(l,'#64748B') for l in lbl_cnt.index], alpha=0.85)
for i,(l,v) in enumerate(lbl_cnt.items()):
    axes[2].text(i, v+20, f'{v/len(df)*100:.1f}%', ha='center', fontweight='bold')
axes[2].set_title('Label Engagement (FIXED)', fontweight='bold')

plt.suptitle('Verifikasi Hasil Ekstraksi Fitur — Head Pose Fixed v3',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/content/verifikasi_headpose_fix.png', dpi=200, bbox_inches='tight')
plt.show()
print('\n✅ Gambar verifikasi tersimpan')

## LANGKAH 9: Download

In [ ]:
from google.colab import files
import shutil

# Simpan ke Drive (sudah otomatis karena OUTPUT_CSV di Drive)
# Download ke lokal
local_csv = '/content/camera_features_real_v3.csv'
shutil.copy(OUTPUT_CSV, local_csv)
files.download(local_csv)
files.download('/content/verifikasi_headpose_fix.png')
print(f'✅ Download selesai!')
print(f'\nFile yang dihasilkan:')
print(f'  camera_features_real_v3.csv  ← upload ke sini untuk update Bab IV')
print(f'  verifikasi_headpose_fix.png  ← cek apakah label engagement sudah normal')